In [36]:
import pandas as pd
import numpy as np
import joblib
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# 1. LOAD DATA
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['price'] = housing.target

# 2. FEATURE ENGINEERING (Making the data smarter)
# We create ratios that matter more to house prices than raw totals
df['rooms_per_household'] = df['AveRooms'] / df['AveOccup']
df['bedrooms_per_room'] = df['AveBedrms'] / df['AveRooms']

# 3. DATA CLEANING
# Remove houses at the $500k cap to stop the model from getting confused
df = df[df['price'] < 5.0]

# 4. DEFINE X AND Y
X = df.drop('price', axis=1).values
y = df['price'].values

# 5. SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. SCALE (Level the playing field)
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

# 7. TRAIN THE COMPLEX MODEL (Random Forest)
# We increased n_estimators to 200 for more "opinions" from the trees
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train_scaled, y_train)

# 8. EVALUATE
score = model.score(X_test_scaled, y_test)
print(f"Improved R-squared Score: {score:.4f}")

# 9. SAVE FOR LATER
joblib.dump(model, 'housing_model_v2.pkl')
joblib.dump(scalar, 'housing_scaler_v2.pkl')

# 10. TEST PREDICTION
# 1. Grab the first 10 houses from the test set
test_samples = X_test_scaled[0:10]
actual_prices = y_test[0:10]

# 2. Get predictions for all 10 at once
predictions = model.predict(test_samples)

print(f"{'House #':<10} | {'Predicted Price':<20} | {'Actual Price':<20} | {'Difference'}")
print("-" * 75)

for i in range(10):
    pred_val = predictions[i] * 100000
    act_val = actual_prices[i] * 100000
    diff = pred_val - act_val

    print(f"House {i+1:<4} | ${pred_val:>14,.2f} | ${act_val:>14,.2f} | ${abs(diff):>12,.2f}")

Improved R-squared Score: 0.7757
House #    | Predicted Price      | Actual Price         | Difference
---------------------------------------------------------------------------
House 1    | $    332,068.50 | $    329,800.00 | $    2,268.50
House 2    | $    296,847.50 | $    294,700.00 | $    2,147.50
House 3    | $    183,724.00 | $    195,700.00 | $   11,976.00
House 4    | $    219,770.00 | $    161,500.00 | $   58,270.00
House 5    | $    370,200.50 | $    275,000.00 | $   95,200.50
House 6    | $    175,221.50 | $    137,000.00 | $   38,221.50
House 7    | $    273,993.50 | $    266,000.00 | $    7,993.50
House 8    | $    107,795.00 | $     81,300.00 | $   26,495.00
House 9    | $    136,904.00 | $     92,800.00 | $   44,104.00
House 10   | $    309,394.00 | $    273,700.00 | $   35,694.00


In [34]:
from sklearn.linear_model import LinearRegression

# 1. Train the Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 2. Get predictions for both
lr_preds = lr_model.predict(X_test_scaled[0:10])
rf_preds = model.predict(X_test_scaled[0:10]) # This uses your RF from the first cell

print(f"{'House #':<8} | {'Actual Price':<14} | {'RF Error':<12} | {'LR Error':<12} | {'Winner'}")
print("-" * 70)

for i in range(10):
    act = y_test[i] * 100000
    rf_err = (rf_preds[i] * 100000) - act
    lr_err = (lr_preds[i] * 100000) - act

    # Determine which model was closer to the truth
    winner = "Random Forest" if abs(rf_err) < abs(lr_err) else "Linear Reg"

    print(f"{i+1:<8} | ${act:>12,.0f} | ${rf_err:>11,.0f} | ${lr_err:>11,.0f} | {winner}")

House #  | Actual Price   | RF Error     | LR Error     | Winner
----------------------------------------------------------------------
1        | $     329,800 | $      2,268 | $    -44,019 | Random Forest
2        | $     294,700 | $      2,147 | $    -19,035 | Random Forest
3        | $     195,700 | $    -11,976 | $     77,726 | Random Forest
4        | $     161,500 | $     58,270 | $     12,425 | Linear Reg
5        | $     275,000 | $     95,200 | $        975 | Linear Reg
6        | $     137,000 | $     38,221 | $     41,207 | Random Forest
7        | $     266,000 | $      7,993 | $     16,942 | Random Forest
8        | $      81,300 | $     26,495 | $    -42,971 | Random Forest
9        | $      92,800 | $     44,104 | $     72,371 | Random Forest
10       | $     273,700 | $     35,694 | $     28,523 | Linear Reg


In [25]:
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847
